# 📑 PageIndex - Vectorless RAG Crash Course
### Reasoning-based RAG with No Vector DB, No Chunking

---

## 🔑 Key Concept

> **Traditional RAG** → chunk → embed → cosine similarity → retrieve  
> **PageIndex RAG** → build tree → LLM reasons over tree → retrieve exact sections

**The problem with vector RAG:**  
`Similarity ≠ Relevance`  
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.

The PageIndex pipeline is organized into three steps:
1. **Document Indexing** — submit a PDF and build a hierarchical tree index
2. **Structure-aware Retrieval** — LLM reasons over the tree to find & fetch relevant sections
3. **Context-aware Reasoning** — generate answers from the retrieved context

---
## 📦 Section 1: Install & Setup

**What we do here:**
- Install PageIndex SDK + Groq
- Load API keys from `.env`
- Initialize the Groq client

> 🔑 Get your **PageIndex API key** from: https://dash.pageindex.ai/api-keys  
> 🔑 Get your **Groq API key** from: https://console.groq.com/keys

In [2]:
# Install required packages
!pip install -U pageindex groq python-dotenv

In [3]:
import os
import json
import time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "Yes" if PAGEINDEX_API_KEY else "No")
print("Groq key loaded:", "Yes" if GROQ_API_KEY else "No")

PageIndex key loaded: Yes
Groq key loaded: Yes


In [4]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

print("PageIndex client ready")
print("Groq client ready")

PageIndex client ready
Groq client ready


---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload the PDF to the PageIndex cloud.
2. PageIndex uses an LLM to understand the document's structure.
3. It builds a hierarchical **tree index**, similar to a smart Table of Contents.
4. It returns a `doc_id` for future operations.

**Why NO chunking?**

Instead of splitting the document into arbitrary 500-token chunks, PageIndex preserves the document's natural structure — chapters, sections, sub-sections, and paragraphs — as defined by the original document.

In [5]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace this with the path to your PDF file.
# Good candidates include annual reports, research papers, legal documents, and textbooks.

PDF_PATH = "attention.pdf"

print(f"Uploading: {PDF_PATH}")

result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print("Upload completed.")
print(f"Document ID: {doc_id}")
print("Save this ID — you'll use it throughout the notebook.")

Uploading: attention.pdf
Upload completed.
Document ID: pi-cmuw88sbp009b09mbl7apqnnp
Save this ID — you'll use it throughout the notebook.


C:\Users\devgo\AppData\Local\Temp\ipykernel_4076\2203039040.py:9: UserWarning: Document "attention.pdf" was stored as "attention_1.pdf".
  result = pi_client.submit_document(PDF_PATH)


In [6]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree index asynchronously.
# Processing time depends on the document size.

print("Building tree index...")
print("This runs once per document; the index is cached for reuse.")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")

    print(f"Status: {status}")

    if status == "completed":
        print("\nTree index ready.")
        break

    if status == "failed":
        print("\nProcessing failed. Check your PDF format.")
        break

    time.sleep(5)

Building tree index...
This runs once per document; the index is cached for reuse.
Status: queued
Status: processing
Status: processing
Status: processing
Status: completed

Tree index ready.


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Abstract
├── 1 Introduction
├── 2 Background
├── 3 Model Architecture
│   ├── 3.1 Encoder and Decoder Stacks
│   ├── 3.2 Attention
│   ├── 3.3 Applications of Attention
│   └── 3.4 Positional Encoding
├── 4 Why Self-Attention
├── 5 Training
├── 6 Results
└── 7 Conclusion
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**PageIndex preserves the document's natural hierarchy instead of breaking it into fixed-size chunks.**

In [7]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"Top-level sections: {len(pageindex_tree)}")
print("\nRaw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top-level sections: 1

Raw tree (first node):
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 15,
  "summary": "This section, titled \"Attention Is All You Need,\" introduces the Transformer model\u2014a novel neural network architecture for sequence transduction based entirely on attention mechanisms, eliminating traditional recurrence and convolutions. It includes author affiliations, an abstract, an introduction to sequence modeling limitations, background on prior models, and a detailed breakdown of the Transformer's encoder-decoder architecture, attention mechanisms, and positional encodings. Furthermore, it compares self-attention to recurrent and convolutional layers, outlines the training setup and datasets (WMT 2014 English-German and English-French), presents state-of-the-art translation and parsing results alongside ablation studies, offers conclusions and future research directions, provides reference materials, and includes

In [8]:
# ── Pretty-print the full tree ───────────────────────────────────────────────

def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page = node.get("page_index", "?")

        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")

        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)


print("Full Document Structure:\n")
print_tree(pageindex_tree)

Full Document Structure:

[0000] Attention Is All You Need (p.?)
  └─ [0001] Abstract (p.?)
  └─ [0002] 1 Introduction (p.?)
  └─ [0003] 2 Background (p.?)
  └─ [0004] 3 Model Architecture (p.?)
    └─ [0005] 3 Model Architecture (intro) (p.?)
    └─ [0006] 3.1 Encoder and Decoder Stacks (p.?)
    └─ [0007] 3.2 Attention (p.?)
      └─ [0008] 3.2 Attention (intro) (p.?)
      └─ [0009] 3.2.1 Scaled Dot-Product Attention (p.?)
      └─ [0010] 3.2.2 Multi-Head Attention (p.?)
      └─ [0011] 3.2.3 Applications of Attention in our Model (p.?)
    └─ [0012] 3.3 Position-wise Feed-Forward Networks (p.?)
    └─ [0013] 3.4 Embeddings and Softmax (p.?)
    └─ [0014] 3.5 Positional Encoding (p.?)
  └─ [0015] 4 Why Self-Attention (p.?)
  └─ [0016] 5 Training (p.?)
  └─ [0017] 6 Results (p.?)
    └─ [0018] 6.1 Machine Translation (p.?)
    └─ [0019] 6.2 Model Variations (p.?)
    └─ [0020] 6.3 English Constituency Parsing (p.?)
  └─ [0021] 7 Conclusion (p.?)
  └─ [0022] References (p.?)
  └─ [002

In [9]:
# ── Count total nodes ────────────────────────────────────────────────────────

def count_nodes(nodes):
    total = len(nodes)

    for node in nodes:
        if node.get("nodes"):
            total += count_nodes(node["nodes"])

    return total


total = count_nodes(pageindex_tree)

print(f"Total nodes in tree: {total}")
print("Each node represents a retrievable section of the document.")

Total nodes in tree: 24
Each node represents a retrievable section of the document.


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**

In [10]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query and document tree to an LLM.
    The LLM reasons over the structure and returns relevant node IDs.

    Returns:
        dict with 'thinking' and 'node_list'
    """

    # Compress tree to reduce token usage - only send titles + short summaries
    def compress(nodes):
        compressed = []

        for node in nodes:
            entry = {
                "node_id": node["node_id"],
                "title": node["title"],
                "summary": node.get("text", "")[:150]
            }

            if node.get("nodes"):
                entry["children"] = compress(node["nodes"])

            compressed.append(entry)

        return compressed

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure, similar to a Table of Contents.

Your task is to identify which node IDs are most likely to contain the answer to the query.

Query:
{query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<brief reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

In [11]:
# ── Test with a sample query ─────────────────────────────────────────────────

query = "What is the architecture of the Transformer model?"

print(f"Query: {query}\n")

result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("Selected Node IDs:", result.get("node_list", []))

Query: What is the architecture of the Transformer model?

LLM Reasoning:
The query asks for the overall architecture of the Transformer. The paper’s architecture description is under the “3 Model Architecture” section and its subsections that detail encoder‑decoder stacks, attention mechanisms, feed‑forward networks, embeddings, and positional encodings.

Selected Node IDs: ['0004', '0005', '0006', '0007', '0012', '0013', '0014']


---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks

In [12]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [23]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(
    query: str,
    nodes: list,
    model: str = "openai/gpt-oss-120b"
) -> str:
    """
    Generates a grounded answer from retrieved PageIndex nodes.

    The answer:
    - Uses only retrieved context
    - Includes section and page citations
    - Uses a clean citation format
    - Preserves mathematical notation in LaTeX
    """

    if not nodes:
        return "No relevant sections found in the document."

    # Build context from retrieved nodes
    context_parts = []

    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst.

Answer the question using ONLY the provided context.

For every factual claim, cite the relevant section title and page number.

Use this exact citation format:
[Section: 3.2.2 Multi-Head Attention | Page ?]

Citation rules:
- Use square brackets [ ] for citations.
- Do NOT use citation brackets such as 【 】.
- Do NOT use Unicode citation markers.
- Do NOT create any other citation format.
- If the page number is unavailable, use "Page ?".
- Do NOT invent page numbers.
- Keep mathematical notation in standard LaTeX.
- Do not add information that is not supported by the provided context.
- Be concise and precise.

Question:
{query}

Context:
{context}

Answer:"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [24]:
# The complete Vectorless RAG function

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:

    Step 1: LLM Tree Search      -> finds relevant node IDs
    Step 2: Node Retrieval       -> fetches section content
    Step 3: Answer Generation    -> produces a cited answer
    """

    if verbose:
        print("=" * 55)
        print(f"Query: {query}")
        print("=" * 55)

    # Step 1: Tree Search
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])

    if verbose:
        print(f"\nReasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"Sections found: {[node['title'] for node in nodes]}")

    # Step 3: Generate answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\nAnswer:\n{answer}")

    return answer

In [25]:
# Run the full pipeline

answer = vectorless_rag(
    query="What is the architecture of the Transformer model?",
    tree=pageindex_tree
)

Query: What is the architecture of the Transformer model?

Reasoning: The Transformer’s architecture is detailed in the “3 Model Architecture” section and its sub‑sections that cover the overall encoder‑decoder design, attention mechanisms, feed‑forward networks, embedd...
Retrieved node IDs: ['0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014']
Sections found: ['3 Model Architecture', '3 Model Architecture (intro)', '3.1 Encoder and Decoder Stacks', '3.2 Attention', '3.2 Attention (intro)', '3.2.1 Scaled Dot-Product Attention', '3.2.2 Multi-Head Attention', '3.2.3 Applications of Attention in our Model', '3.3 Position-wise Feed-Forward Networks', '3.4 Embeddings and Softmax', '3.5 Positional Encoding']

Answer:
The Transformer is built as a **standard encoder‑decoder sequence‑to‑sequence architecture** in which both the encoder and the decoder consist of a stack of **six identical layers** [Section: 3 Model Architecture (intro) | Page ?].

**Encoder*

In [26]:
# Test with multiple queries

test_queries = [
    "What is the architecture of the Transformer model?",
    "How does scaled dot-product attention work?",
    "Why does the Transformer use positional encoding?"
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)

    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What is the architecture of the Transformer model?
A: The Transformer follows an encoder‑decoder architecture in which an **encoder** maps an input sequence of symbols \((x_1,\dots ,x_n)\) to a continuous representation \(\mathbf{z}\), and a **decoder** generates the output sequence \((y_1,\dots ,y_m)\) one token at a time in an auto‑regressive fashion...
-------------------------------------------------------

Q: How does scaled dot-product attention work?
A: Scaled dot‑product attention takes a set of queries \(Q\), keys \(K\) and values \(V\) (with query/key dimension \(d_k\) and value dimension \(d_v\)). It first computes the dot products between each query and all keys, forming the matrix \(QK^{\mathsf{T}}\). Each dot product is then scaled by the fa...
-------------------------------------------------------

Q: Why does the Transformer use positional encoding?
A: The Transformer has no recurrence or convolution, so it cannot infer token order on its own. Positional encodings 

---
## 🎓 Section 6: Expert-Guided Retrieval

**PageIndex can incorporate domain-specific retrieval rules directly into the prompt.**

With vector RAG, changing retrieval behavior often requires modifying the embedding or retrieval pipeline.

With PageIndex, you can guide the LLM's tree search using simple rules:

```text
"If the query is about attention → prioritize Section 3.2"
"If the query is about positional encoding → prioritize Section 3.5"
"If the query is about training → prioritize Section 5"

In [27]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These rules guide the LLM toward relevant sections for specific queries.

EXPERT_RULES = """
Expert routing rules for the "Attention Is All You Need" paper:
- Transformer architecture queries      → Section 3: Model Architecture
- Encoder/decoder queries               → Section 3.1: Encoder and Decoder Stacks
- Attention queries                     → Section 3.2: Attention
- Scaled dot-product attention queries  → Section 3.2.1
- Multi-head attention queries          → Section 3.2.2
- Attention applications queries        → Section 3.2.3
- Feed-forward network queries          → Section 3.3
- Embeddings and softmax queries        → Section 3.4
- Positional encoding queries            → Section 3.5
- Self-attention advantages              → Section 4: Why Self-Attention
- Training-related queries              → Section 5: Training
- Results or performance queries         → Section 6: Results
- Conclusion-related queries             → Section 7: Conclusion
"""

print("Expert routing rules defined.")
print("These rules can be injected into the retrieval prompt at query time.")

Expert routing rules defined.
These rules can be injected into the retrieval prompt at query time.


In [28]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
# Krish Naik Academy | 21 Modules | 38 Sections | 481 Topics

EXPERT_ROUTING_RULES = """
Route queries to the correct module using these rules:

M1  Neural Network Refresher → backprop, activations, optimizers, PyTorch basics

M2  Hardware                 → GPU, TPU, Apple Silicon, compute infrastructure

M3  Transformers 101         → attention, self-attention, encoder-decoder, MHA

M4  Tokenization             → BPE, WordPiece, SentencePiece, Byte Latent Transformers

M5  Finetuning Architectures → hands-on BERT/GPT/T5 finetuning, Hugging Face

M6  KV Cache & Attention     → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM

M7  Scaling Laws             → Kaplan, Chinchilla, compute-optimal training

M8  Mixture of Experts       → MoE, sparse computation, Mixture of Depths

M9  Modern LLM Finetuning    → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO, 
                                quantization, TRL, Unsloth, synthetic data, 
                                reasoning models, evaluation, deployment

M10 SLM                      → small language models, pruning, when SLM vs LLM

M11 Knowledge Distillation   → student-teacher, soft labels, DistilBERT, DeepSeek-R1

M12 Hybrid Architectures     → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers

M13 Vision Foundations       → ViT, patch embeddings, CLIP, SigLIP, DINOv2

M14 Visual Language Models   → VLM architecture, aligner, multimodal reasoning

M15 Stable Diffusion & DiT   → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth

M16 Embedding Models         → dense, sparse, binary, Matryoshka, MRL, fine-tuning

M17 RAG                      → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                self/corrective/adaptive/agentic RAG, Graph RAG,
                                multi-modal RAG, ColPali, RAG security

M18 Context Engineering      → prompt vs context engineering, memory architecture,
                                context compression, KV cache, agent context lifecycle

M19 DSPy                     → signatures, modules, MIPROv2, self-optimizing RAG

M20 Agents                   → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                A2A, guardrails, observability, evaluation

M21 RL                       → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                RLHF vs RLVR, policy gradient, DeepSeek-R1 training

Cross-cutting rules:

- "learning path / where to start"    → M1 → M2 → M3 in order

- "production / deployment / serving" → M9 (quantization) + M20 (agents)

- "fine-tuning vs RAG"                → M9 + M17 + M18

- "multimodal / vision + language"    → M13 + M14 + M17 (multi-modal RAG)

- "reasoning models / test-time RL"   → M9 (reasoning) + M21 (GRPO/DAPO)
"""

print("Expert routing rules defined.")
print("These rules can be injected into the retrieval prompt at query time.")

Expert routing rules defined.
These rules can be injected into the retrieval prompt at query time.


In [29]:
# ── Expert-guided tree search ────────────────────────────────────────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "openai/gpt-oss-120b"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """

    def compress(nodes):
        compressed = []

        for node in nodes:
            entry = {
                "node_id": node["node_id"],
                "title": node["title"],
                "page": node.get("page_index", "?"),
                "summary": node.get("text", "")[:150]
            }

            if node.get("nodes"):
                entry["children"] = compress(node["nodes"])

            compressed.append(entry)

        return compressed

    compressed_tree = compress(tree)

    prompt = f"""You are a domain expert analyzing a document.

Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query:
{query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Expert Routing Rules:
{expert_rules}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

In [30]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────

query = "How does multi-head attention work in the Transformer?"

print(f"Query: {query}\n")

# Without expert rules
print("Without Expert Rules")
basic = llm_tree_search(query, pageindex_tree)

print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("With Expert Rules")
guided = llm_tree_search_with_expert(
    query,
    pageindex_tree,
    EXPERT_ROUTING_RULES
)

print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

Query: How does multi-head attention work in the Transformer?

Without Expert Rules
Nodes: ['0010', '0008', '0009']

With Expert Rules
Nodes: ['0010']
Reasoning: The query asks for an explanation of multi-head attention in the Transformer. According to the routing rules, this belongs to module M3 (Transformers 101). In the document tree, the section titled '3.2.2 Multi-Head Attention' (node_id 0010) directly covers this topic, making it the most relevant nod


In [31]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result = llm_tree_search_with_expert(query, tree, rules)
    nodes = find_nodes_by_ids(tree, result.get("node_list", []))

    return generate_answer(query, nodes)


# Run it
answer = expert_rag(
    query="How does multi-head attention work in the Transformer?",
    tree=pageindex_tree,
    rules=EXPERT_ROUTING_RULES
)

print(answer)

Multi‑head attention first linearly projects the original queries \(Q\), keys \(K\) and values \(V\) into \(h\) separate subspaces using learned matrices \(W_i^{Q},W_i^{K},W_i^{V}\) (each mapping from \(d_{\text{model}}\) to \(d_k\) or \(d_v\)) [Section: 3.2.2 Multi-Head Attention | Page ?].  
For each head \(i\) the standard attention function is applied in parallel to the projected tensors, producing a \(d_v\)‑dimensional output \(\text{head}_i = \text{Attention}(QW_i^{Q},KW_i^{K},VW_i^{V})\) [Section: 3.2.2 Multi-Head Attention | Page ?].  
The \(h\) head outputs are then concatenated and linearly transformed by a final matrix \(W^{O}\) to obtain the multi‑head result:  

\[
\text{MultiHead}(Q,K,V)=\text{Concat}(\text{head}_1,\dots,\text{head}_h)\,W^{O}.
\]

Because each head operates on a lower‑dimensional subspace (\(d_k = d_v = d_{\text{model}}/h\)), the total computational cost is comparable to a single full‑dimensional attention layer while allowing the model to “jointly attend

---
## 💬 Section 7: Chat API — Zero LLM Setup

**When to use this:**
- You don't want to manage LLM API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat application and want PageIndex to handle the LLM interaction

PageIndex provides a built-in Chat API, so you can ask questions about your document by passing the `doc_id` and your query.

**Note:** This approach is different from the Groq-based RAG pipeline used earlier. Here, PageIndex handles the LLM interaction for you.

In [32]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex handles the LLM interaction internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[
        {"role": "user", "content": question}
    ],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]

print("Chat API Answer:")
print(answer)

Chat API Answer:
I'll look at this document's structure first, then pull the key sections.This is "Attention Is All You Need" (the Transformer paper). Key findings:

**1. Attention alone is sufficient — recurrence and convolution are unnecessary**
The Transformer is the first sequence transduction model built entirely on attention, replacing recurrent layers with multi-head self-attention plus position-wise feed-forward layers (p. 2–6, 10).

**2. New state-of-the-art in machine translation at a fraction of the cost** (Table 2, p. 8)
- WMT 2014 EN→DE: **28.4 BLEU** (big model), beating all prior models *including ensembles* by >2.0 BLEU
- WMT 2014 EN→FR: **41.8 BLEU**, best published single model at <1/4 the training cost of the prior SOTA
- Even the base model (27.3 BLEU, 3.3·10¹⁸ FLOPs) surpasses all previously published models/ensembles
- Big model trained in just 3.5 days on 8 P100 GPUs; base model in 12 hours

**3. Self-attention wins on complexity, parallelism, and path length** (

In [33]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Maintain conversation history across multiple questions

conversation_history = []


def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document while maintaining conversation history."""
    
    global conversation_history

    # Add the user's message to the conversation
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Send the complete conversation history to PageIndex
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )

    # Extract the assistant's response
    assistant_reply = response["choices"][0]["message"]["content"]

    # Add the response to the conversation history
    conversation_history.append({
        "role": "assistant",
        "content": assistant_reply
    })

    return assistant_reply


# Simulate a 3-turn conversation about the Transformer paper
questions = [
    "What is the main idea behind the Transformer architecture?",
    "How does multi-head attention contribute to this architecture?",
    "Why does the Transformer use positional encoding?"
]

for question in questions:
    print(f"\nUser: {question}")

    reply = chat_with_doc(question, doc_id)

    print(f"Assistant: {reply}")
    print("-" * 55)


User: What is the main idea behind the Transformer architecture?
Assistant: I'll look at the paper's core sections.# The Core Idea: "Attention Is All You Need"

**The central claim:** you can build a sequence transduction model using *only* attention mechanisms — dispensing with recurrence and convolutions entirely (p. 1).

## The problem it solves

RNNs process sequences position-by-position, generating hidden state $h_t$ from $h_{t-1}$. This "inherently sequential nature precludes parallelization within training examples," which becomes critical at long sequence lengths (p. 2). Convolutional alternatives (ByteNet, ConvS2S) parallelize, but the number of operations needed to relate two distant positions grows with their distance — logarithmically or linearly — making long-range dependencies hard to learn (p. 2).

## The insight

**Self-attention** relates every position in a sequence to every other position directly, in one step. This reduces the path length between any two positions

---
## 🛠️ Section 8: Self-Hosted Open Source Option

**Use this when:**
- You don't want to send documents to the cloud
- You need full data privacy or on-premises deployment
- You want to inspect or customize the tree-building pipeline

The open-source PageIndex repository is available at:

https://github.com/VectifyAI/PageIndex

It allows you to run the PageIndex pipeline locally using your own LLM API key.

**What the CLI does:**
1. Reads your PDF
2. Detects the existing Table of Contents, if available
3. Uses an LLM to build the hierarchical tree
4. Saves a `document_name_pageindex.json` file alongside the PDF

> **Coming Soon:**  
> This section will cover how to run PageIndex locally using the open-source implementation, including local document processing and self-hosted tree generation.

---
## 🧹 Section 9: Cleanup

Delete documents from the PageIndex cloud when you're done  
to keep your storage clean.

In [ ]:
# ── Delete document from cloud ───────────────────────────────────────────────
# WARNING: This permanently deletes the document and its tree index.
# Uncomment the lines below when you are finished with this document.

# pi_client.delete_document(doc_id)
# print(f"Deleted document: {doc_id}")

print("Document deletion is commented out.")

---
## ✅ Summary

You've now built a complete **Vectorless RAG** system using PageIndex and Groq.

### What you built:

1. **`llm_tree_search()`** — Uses an LLM to reason over the document tree and identify relevant nodes
2. **`find_nodes_by_ids()`** — Retrieves the actual content from the selected nodes
3. **`generate_answer()`** — Generates grounded answers using the retrieved document content with section and page citations
4. **`vectorless_rag()`** — Combines tree search, node retrieval, and answer generation into a complete pipeline
5. **`expert_rag()`** — Adds domain-specific routing rules to guide the retrieval process without fine-tuning
6. **Chat API** — Provides a simple multi-turn Q&A interface over the indexed document

### Key Takeaways:

- **Similarity search is not the only way to retrieve relevant information** — PageIndex uses the document's hierarchical structure for retrieval
- **Tree-based retrieval** preserves the document's structure and makes retrieved sections traceable
- **LLM-based tree search** allows the model to reason over the document's organization rather than relying only on embedding similarity
- **Expert routing rules** can guide retrieval behavior through prompt-based instructions without additional model training
- **PageIndex Chat API** provides a convenient way to interact with an indexed document without implementing the complete retrieval pipeline manually

### Document Used:

**Attention Is All You Need** — the Transformer architecture paper.

The notebook demonstrates how PageIndex can build a hierarchical representation of the paper and use that structure for question answering.

---

### Resources

- **GitHub:** https://github.com/VectifyAI/PageIndex
- **Documentation:** https://docs.pageindex.ai
- **Chat Platform:** https://chat.pageindex.ai
- **Blog:** https://pageindex.ai/blog/pageindex-intro